# Generating QRC inputs from an ASE / MLIP frequency calculation

pyQRC normally reads a Gaussian, ORCA, or Q-Chem output file. When the Hessian comes from a machine-learned interatomic potential (MLIP) driven through [ASE](https://wiki.fysik.dtu.dk/ase/) there is no such file — so the helper [`ase2gaussian.py`](ase2gaussian.py) in this directory writes the geometry, frequencies, and normal modes in the Gaussian log format that cclib (and therefore pyQRC) parses. From there, everything works exactly as with a real QM frequency job.

This notebook uses the [MACE-OFF](https://github.com/ACEsuit/mace) foundation model for organic molecules, but any ASE calculator that supports forces works the same way (ANI, AIMNet2, xTB, SO3LR, ...).

Requirements: `pip install pyqrc ase mace-torch`

## Setup

We work in a `scratch/` subdirectory (ignored by git) since pyQRC writes its files next to the log it reads. The thread settings avoid an OpenMP crash with PyTorch on macOS and must come before the first `torch` import.

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
from ase import Atoms
from ase.constraints import FixedPlane
from ase.optimize import LBFGS
from ase.vibrations import Vibrations

sys.path.insert(0, str(Path.cwd()))
from ase2gaussian import extract_vibrations, write_gaussian_freq_log

HERE = Path.cwd()
SCRATCH = HERE / "scratch"
if SCRATCH.exists():
    shutil.rmtree(SCRATCH)
SCRATCH.mkdir()
os.chdir(SCRATCH)


def run_pyqrc(*args):
    """Run the pyqrc command line inside scratch/, echoing its output."""
    result = subprocess.run(
        [sys.executable, "-m", "pyqrc", *args],
        capture_output=True, text=True, cwd=SCRATCH,
    )
    print(result.stdout, end="")
    if result.returncode != 0:
        print(result.stderr, end="")
    return result.returncode


def show(filename, n=None):
    """Print the first n lines (default: all) of a generated file."""
    lines = (SCRATCH / filename).read_text().splitlines()
    print("\n".join(lines if n is None else lines[:n]))

## Step 1: a transition state from an MLIP

Planar NH$_3$ is the transition state for umbrella inversion. We optimize it with MACE-OFF under a planarity constraint (a poor man's saddle-point search — for real systems use a TS optimizer like Sella), then release the constraint for the frequency calculation.

In [2]:
from mace.calculators import mace_off

calc = mace_off(model="small", default_dtype="float64", device="cpu")

atoms = Atoms("NH3", positions=[[0.00, 0.000, 0.0],
                                [1.02, 0.000, 0.0],
                                [-0.51, 0.883, 0.0],
                                [-0.51, -0.883, 0.0]])
atoms.calc = calc
atoms.set_constraint([FixedPlane(i, (0, 0, 1)) for i in range(len(atoms))])
LBFGS(atoms, logfile=None).run(fmax=0.001)
atoms.set_constraint()
e_ts = atoms.get_potential_energy()
print(f"planar NH3: E = {e_ts:.6f} eV, N-H = {atoms.get_distance(0, 1):.4f} A")

cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
Using MACE-OFF23 MODEL for MACECalculator with /Users/rpaton/.cache/mace/MACE-OFF23_small.model
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.
planar NH3: E = -1539.861421 eV, N-H = 0.9963 A


## Step 2: frequencies and normal modes

`ase.vibrations.Vibrations` builds the Hessian by finite differences of the MLIP forces. `extract_vibrations` converts ASE's complex energies to the Gaussian sign convention (negative = imaginary) and drops the six near-zero translation/rotation modes.

In [3]:
vib = Vibrations(atoms)
vib.run()
frequencies, modes = extract_vibrations(vib)
print("frequencies (cm-1):", np.round(frequencies, 1))

frequencies (cm-1): [-826.3 1543.7 1544.2 3685.1 3853.7 3854.1]


Exactly one imaginary frequency — the umbrella inversion mode — as expected for a first-order saddle point.

## Step 3: write the Gaussian-style log

`write_gaussian_freq_log` emits only the blocks cclib and pyQRC actually read. The `route` argument is echoed into the inputs pyQRC generates; `# opt` suits the usual displace-then-reoptimize workflow.

In [4]:
EV_TO_HARTREE = 1 / 27.211386245988
write_gaussian_freq_log("nh3_ts_mace.log", atoms, frequencies, modes,
                        energy=e_ts * EV_TO_HARTREE, route="# opt")
show("nh3_ts_mace.log", n=22)

 Entering Gaussian System, Link 0=g16
 This file was written by ase2gaussian.py (pyQRC examples), not by
 Gaussian. It mimics the output format of Gaussian, Inc. so that
 cclib-based tools can read ASE/MLIP frequency results.
 ----------------------------------------------------------------------
 # opt
 ----------------------------------------------------------------------
 Charge =  0 Multiplicity = 1
 NAtoms=     4 NActive=     4
                         Standard orientation:                         
 ---------------------------------------------------------------------
 Center     Atomic      Atomic             Coordinates (Angstroms)
 Number     Number       Type             X           Y           Z
 ---------------------------------------------------------------------
      1          7           0        0.000037    0.000000    0.000000
      2          1           0        0.996353    0.000000    0.000000
      3          1           0       -0.498195    0.862772    0.000000
 

## Step 4: run pyQRC on it

By default pyQRC displaces along all imaginary modes — here, the single inversion mode. The benchmark in the README found an amplitude of **0.3** performs best.

In [5]:
run_pyqrc("nh3_ts_mace.log", "--amp", "0.3", "--name", "QRC")
show("nh3_ts_mace_QRC.com")

o   nh3_ts_mace.log has 1 imaginary frequencies: processing
%chk=nh3_ts_mace_QRC.chk
%nproc=1
%mem=4GB
# opt


nh3_ts_mace_QRC

0 1
 N   0.00003700   0.00000000   0.03600000
 H   0.99635300   0.00000000  -0.17100000
 H  -0.49819500   0.86277200  -0.17100000
 H  -0.49819500  -0.86277200  -0.17100000



The nitrogen has moved out of the H$_3$ plane. In a QM workflow you would now submit this `.com` file to Gaussian. But since our potential is an MLIP, we can close the loop right here.

## Step 5: reoptimize the displaced geometry with the MLIP

Read the displaced geometry back into ASE and let LBFGS relax it — the QRC step pushes the structure off the saddle point so the optimizer falls into the adjacent minimum (pyramidal NH$_3$).

In [6]:
from ase.io import read

displaced = read("nh3_ts_mace_QRC.com", format="gaussian-in")
displaced.calc = calc
LBFGS(displaced, logfile=None).run(fmax=0.001)
e_min = displaced.get_potential_energy()

n_height = np.linalg.norm(displaced.positions[0] - displaced.positions[1:].mean(axis=0))
print(f"N out-of-plane distance: {n_height:.3f} A (0 = planar)")
print(f"inversion barrier: {(e_ts - e_min) * 23.0605:.1f} kcal/mol")

N out-of-plane distance: 0.376 A (0 = planar)
inversion barrier: 4.9 kcal/mol


The structure relaxed to pyramidal NH$_3$, and the resulting inversion barrier of ~5 kcal/mol is close to the experimental value (~5.8 kcal/mol) — the whole minimum-finding loop ran on the MLIP without a single QM calculation.

## Where the files went

Everything generated above is in the `scratch/` subdirectory — delete it when you are done. For your own systems, the recipe is: any ASE calculator → `Vibrations` → `extract_vibrations` → `write_gaussian_freq_log` → `pyqrc`, with all of pyQRC's options (`--amp`, `--freqnum`, `--auto`, reverse displacement via negative amplitudes) available as usual. See the [project README](../../README.md) for the full option list.